# Cabo de Palos & La Manga
## Day 13 · Interactions · #30DayMapChallenge 2026

Compare how places to eat, grab a coffee, have a drink and go out are distributed in Cabo de Palos and southern La Manga.

**Project by María Victoria León Parra**

Run from the `day13-interactions/` directory using Python 3.11 or later. Install the packages with `python -m pip install -r requirements.txt` and run all cells. The first run needs Overpass access. Later runs reuse the OpenStreetMap extracts in `data/`. The first run also downloads a small OpenFreeMap extract for the background. The exported `index.html` opens directly in a browser and works offline.

In [1]:
# Set up two equal-area study windows and the OSM tags to request.
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import json
import os

import folium
import geopandas as gpd
import osmnx as ox
import pandas as pd
from IPython.display import display, Markdown
from shapely.geometry import box

DATA = Path("data")
DATA.mkdir(exist_ok=True)
TITLE = "Day 13 | Cabo de Palos & La Manga"
FOOTER = "#30DayMapChallenge | 2026 | Day - 13 | Interactions | Cabo de Palos & La Manga | Map by María Victoria León Parra"
# UTM 30N lets us specify identical 1,800 m squares around the two centres.
# The first includes the harbour and village, the second the Plaza Bohemia
# area and both shores of the southern strip. The windows cover water too.
CENTRES = {"cabo": ("Cabo de Palos", 37.6311, -0.6987),
           "manga": ("Southern La Manga", 37.6607, -0.7241)}
WINDOW_METRES = 1800
ZOOM = 15
MOBILE_ZOOM = 14
# Keep the café query separate so its snapshot can be added or refreshed independently.
VENUE_QUERIES = {
    "osm": {"amenity": ["restaurant", "bar", "pub", "nightclub"]},
    "cafes-osm": {"amenity": ["cafe"]},
}
AMENITY_LABELS = {"restaurant": "Restaurant", "bar": "Bar", "pub": "Pub",
                  "nightclub": "Nightclub", "cafe": "Café"}

# For a different region, choose a projected CRS in metres that suits both centres.
centres_utm = gpd.GeoSeries.from_xy(
    [place[2] for place in CENTRES.values()],
    [place[1] for place in CENTRES.values()], crs="EPSG:4326",
).to_crs("EPSG:32630")
half = WINDOW_METRES / 2
windows = gpd.GeoSeries(
    [box(p.x - half, p.y - half, p.x + half, p.y + half) for p in centres_utm],
    index=list(CENTRES), crs=centres_utm.crs,
).to_crs("EPSG:4326")
(DATA / "study-areas.geojson").write_text(
    gpd.GeoDataFrame(
        {"area": [place[0] for place in CENTRES.values()]}, geometry=windows.reset_index(drop=True)
    ).to_json(drop_id=True), encoding="utf-8",
)

696

In [2]:
# Load the saved OSM features, or fetch and save a snapshot for each study window.
from osmnx._errors import InsufficientResponseError, ResponseStatusCodeError
from requests.exceptions import RequestException

ox.settings.overpass_url = os.environ.get("OVERPASS_URL", "https://overpass-api.de/api")
ox.settings.requests_timeout = 45
ox.settings.overpass_rate_limit = False
ox.settings.use_cache = False  # The portable GeoJSON snapshots are the persistent cache.
snapshots = {key: [] for key in CENTRES}
unavailable = []

for key, (label, _, _) in CENTRES.items():
    for suffix, tags in VENUE_QUERIES.items():
        path = DATA / f"{key}-{suffix}.geojson"
        # Check the window and tag metadata so edited settings cannot silently
        # reuse data from the previous location or a different tag query.
        query = {"window": windows[key].wkt, "tags": tags}
        saved = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        if saved.get("query") != query:
            try:
                result = ox.features_from_polygon(windows[key], tags=tags).reset_index()
                # Keep the full geometries and the fields needed to trace each record
                # back to OSM. Restricting the schema keeps snapshots easy to inspect.
                fields = ["element", "id", "amenity", "name", "geometry"]
                for field in fields:
                    if field not in result:
                        result[field] = None
                saved = json.loads(result[fields].to_json(drop_id=True))
            except InsufficientResponseError as error:
                # Treat an explicitly empty Overpass response as a valid empty snapshot.
                if "No matching features" not in str(error):
                    unavailable.append(f"{label} ({suffix})")
                    print(f"OSM data unavailable for {label}: {error}")
                    continue
                saved = {"type": "FeatureCollection", "features": []}
            except (RequestException, OSError, ResponseStatusCodeError) as error:
                unavailable.append(f"{label} ({suffix})")
                print(f"OSM data unavailable for {label}: {error}")
                continue
            saved.update(query=query, retrieved_utc=datetime.now(timezone.utc).isoformat(),
                         endpoint=ox.settings.overpass_url)
            # Replace a completed snapshot atomically so an interrupted write can be retried.
            temporary = path.with_suffix(".tmp")
            temporary.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")
            temporary.replace(path)

        snapshots[key].extend(saved["features"])
        print(f"{label} · {path.name} · OSM snapshot {saved['retrieved_utc'][:10]}")

Cabo de Palos · cabo-osm.geojson · OSM snapshot 2026-11-12
Cabo de Palos · cabo-cafes-osm.geojson · OSM snapshot 2026-11-12
Southern La Manga · manga-osm.geojson · OSM snapshot 2026-11-12
Southern La Manga · manga-cafes-osm.geojson · OSM snapshot 2026-11-12


In [3]:
# Prepare marker positions and choose the category filters using the saved snapshots.
from shapely.geometry import shape

venues = {key: [] for key in CENTRES}
for key, features in snapshots.items():
    seen = set()
    for feature in features:
        properties = feature["properties"]
        amenity = properties["amenity"]
        identity = (properties["element"], properties["id"])
        if identity in seen or amenity not in AMENITY_LABELS or not feature["geometry"]:
            continue
        # Clip mapped venues to the window before choosing a point on the
        # geometry. A representative point also works for building footprints.
        clipped = shape(feature["geometry"]).intersection(windows[key])
        if clipped.is_empty:
            continue
        seen.add(identity)
        name = (properties.get("name") or "").strip()
        point = clipped.representative_point()
        venues[key].append((point.y, point.x, name or "Name not available", amenity))

# Keep one filter per group so the nightclub is distinguishable from
# bars and pubs. Both maps get every layer, even if one is empty.
nightclubs = sum(venue[3] == "nightclub" for area in venues.values() for venue in area)
categories = {
    "restaurants": ("Restaurants", "#b7462e"),
    "cafes": ("Cafés", "#995c00"),
    "bars": ("Bars & Pubs", "#007b74"),
    "nightclubs": ("Nightclubs", "#6f48aa"),
}
layer_for = {"restaurant": "restaurants", "cafe": "cafes", "bar": "bars", "pub": "bars", "nightclub": "nightclubs"}

if unavailable:
    display(Markdown("**OpenStreetMap coverage unavailable** · " + ", ".join(unavailable)))
else:
    counts = pd.DataFrame({
        key: {tag: sum(row[3] == tag for row in venues[key]) for tag in AMENITY_LABELS}
        for key in CENTRES
    }).rename(columns={key: place[0] for key, place in CENTRES.items()})
    display(counts.rename(index=AMENITY_LABELS))
    display(Markdown(
        f"**Nightclubs mapped across both windows · {nightclubs}.**"
    ))

,Cabo de Palos,Southern La Manga
Restaurant,23,7
Bar,11,1
Pub,1,3
Nightclub,1,0
Café,5,1


**Nightclubs mapped across both windows · 1.**

In [4]:
# Save a local vector basemap for the two study areas and their surroundings.
import gzip
from math import floor, pi

import mapbox_vector_tile
import requests
from shapely.geometry import mapping
from shapely.ops import transform

# Request only the tiles intersecting the two windows and a context buffer.
# Web Mercator matches the vector tile grid. The resulting GeoJSON is WGS84,
# like the study windows and venues, so Leaflet can render it directly.
BASEMAP_MARGIN = 2_500  # metres in the local UTM projection
TILE_ZOOM = 14
context = (
    gpd.GeoSeries([box(*windows.to_crs(centres_utm.crs).total_bounds).buffer(
        BASEMAP_MARGIN, cap_style="square"
    ).envelope], crs=centres_utm.crs).to_crs("EPSG:3857").iloc[0].envelope
)
basemap_path = DATA / "basemap.geojson"
basemap_key = {"bounds_3857": list(context.bounds), "tile_zoom": TILE_ZOOM}
basemap = json.loads(basemap_path.read_text(encoding="utf-8")) if basemap_path.exists() else {}

if basemap.get("query") != basemap_key:
    # Resolve a single release to avoid mixing dates within the same basemap.
    metadata = requests.get("https://tiles.openfreemap.org/planet/latest", timeout=30)
    metadata.raise_for_status()
    template = metadata.json()["tiles"][0]

    from pyproj import Transformer

    to_lonlat = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    origin = pi * 6_378_137
    tile_size = 2 * origin / (2 ** TILE_ZOOM)
    west, south, east, north = context.bounds
    x_tiles = range(floor((west + origin) / tile_size), floor((east + origin) / tile_size) + 1)
    y_tiles = range(floor((origin - north) / tile_size), floor((origin - south) / tile_size) + 1)
    styles = {
        "water": {"Polygon", "MultiPolygon"},
        "park": {"Polygon", "MultiPolygon"},
        "building": {"Polygon", "MultiPolygon"},
        "transportation": {"LineString", "MultiLineString"},
    }
    features = []
    for x in x_tiles:
        for y in y_tiles:
            response = requests.get(template.format(z=TILE_ZOOM, x=x, y=y), timeout=30)
            response.raise_for_status()
            payload = response.content
            if payload.startswith(bytes((31, 139))):
                payload = gzip.decompress(payload)
            tile = mapbox_vector_tile.decode(payload)
            left = x * tile_size - origin
            top = origin - y * tile_size
            # Vector tiles include overlapping buffers. Clip to each tile's
            # unbuffered bounds before assembling the basemap to avoid duplication.
            clip = box(left, top - tile_size, left + tile_size, top).intersection(context)
            for layer, geometry_types in styles.items():
                if layer not in tile:
                    continue
                units = tile_size / tile[layer]["extent"]

                def tile_to_mercator(px, py):
                    # The decoder returns upward-increasing y coordinates.
                    return left + px * units, top - tile_size + py * units

                for feature in tile[layer]["features"]:
                    geometry = shape(feature["geometry"])
                    if geometry.geom_type not in geometry_types:
                        continue
                    kind = feature["properties"].get("class", "")
                    if layer == "transportation" and kind in {"rail", "transit", "ferry"}:
                        continue
                    projected = transform(tile_to_mercator, geometry)
                    clipped = projected.buffer(0).intersection(clip) if not projected.is_valid else projected.intersection(clip)
                    if clipped.is_empty or clipped.geom_type not in geometry_types:
                        continue
                    geographic = transform(to_lonlat.transform, clipped.simplify(1))
                    features.append({
                        "type": "Feature", "geometry": mapping(geographic),
                        "properties": {"layer": layer, "class": kind},
                    })
    basemap = {
        "type": "FeatureCollection", "features": features, "query": basemap_key,
        "source": template, "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    }
    temporary = basemap_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(basemap, separators=(",", ":")), encoding="utf-8")
    temporary.replace(basemap_path)
print(f"Local basemap · {len(basemap['features']):,} features · {basemap['retrieved_utc'][:10]}")

Local basemap · 936 features · 2026-11-12


## Map

Explore both places with the shared category filters. The dashed outlines show the equal-sized study windows. On narrow screens, scroll from one map to the other. Both start at the same zoom and can be explored independently.

In [5]:
# Build the Folium layers and give both Leaflet panels the same starting zoom.
from branca.element import Element

figure = folium.Figure(title=TITLE)
vendor = Path("vendor")
for filename, tag in [("leaflet.css", "style"), ("leaflet.js", "script"), ("jquery.min.js", "script")]:
    asset = (vendor / filename).read_text(encoding="utf-8")
    # The source-map directives are only needed for debugging the libraries.
    asset = "\n".join(line for line in asset.splitlines() if not line.startswith("//# sourceMappingURL="))
    figure.header.add_child(Element(f"<{tag}>{asset}</{tag}>"))
for filename in ("leaflet-LICENSE.txt", "jquery-LICENSE.txt"):
    license_text = (vendor / filename).read_text(encoding="utf-8")
    figure.header.add_child(Element(f"<!-- {license_text} -->"))

# Draw water over protected areas that extend across the lagoon. Suppressing strokes
# around filled water polygons hides the boundaries between adjacent tiles.
order = {"park": 0, "water": 1, "building": 2, "transportation": 3}
background = {
    "type": "FeatureCollection",
    "features": sorted(basemap["features"], key=lambda item: order[item["properties"]["layer"]]),
}

def basemap_style(feature):
    layer = feature["properties"]["layer"]
    fills = {"water": "#abd2df", "park": "#dce8d5", "building": "#d9d2c6"}
    if layer in fills:
        return {"stroke": False, "fill": True, "fillColor": fills[layer], "fillOpacity": 1}
    major = feature["properties"]["class"] in {"motorway", "trunk", "primary", "secondary", "tertiary"}
    return {"color": "#c49a63" if major else "#ffffff", "weight": 2.3 if major else 1.3, "opacity": 1}

maps = {}
groups = {}
for key, (label, latitude, longitude) in CENTRES.items():
    panel = folium.Map(
        location=[latitude, longitude], zoom_start=ZOOM, tiles=None, min_zoom=MOBILE_ZOOM, max_zoom=19,
        control_scale=False, zoom_control=False, scroll_wheel_zoom=False,
    )
    # The dependencies are embedded below so local file previews and
    # GitHub Pages use the same self-contained HTML.
    panel.default_js = []
    panel.default_css = []
    folium.GeoJson(
        background, name="Local basemap", style_function=basemap_style, interactive=False,
    ).add_to(panel)
    folium.GeoJson(
        windows[key].__geo_interface__, name="Study window",
        style_function=lambda _: {"color": "#426279", "dashArray": "5 5",
                                  "weight": 2, "fill": False, "interactive": False},
    ).add_to(panel)
    maps[key] = panel
    groups[key] = {
        category: folium.FeatureGroup(name=legend, control=False).add_to(panel)
        for category, (legend, _) in categories.items()
    }
    for lat, lon, name, amenity in venues[key]:
        category = layer_for[amenity]
        _, color = categories[category]
        # OSM names are user-supplied text. Escape HTML and the characters
        # that could be interpreted inside Folium's JavaScript template literal.
        safe_name = escape(name).replace("`", "&#96;").replace("$", "&#36;").replace("\\", "&#92;")
        popup = f"<strong>{safe_name}</strong><br>{AMENITY_LABELS[amenity]}"
        folium.CircleMarker(
            [lat, lon], radius=6, color="#ffffff", weight=1.5,
            fill=True, fill_color=color, fill_opacity=0.94,
            popup=folium.Popup(popup, max_width=240),
        ).add_to(groups[key][category])

In [6]:
# Lay out the comparison page and connect the controls to both Leaflet maps.
# A single set of native checkboxes doubles as the legend. Each checkbox
# owns one FeatureGroup on each map, so changes always apply to both.
filters = "".join(
    f'<label><input type="checkbox" value="{key}" checked>'
    f'<span class="swatch" style="--marker:{color}" aria-hidden="true"></span>{escape(label)}</label>'
    for key, (label, color) in categories.items()
)
status = (
    '<p class="notice" role="status">OpenStreetMap venue data unavailable for '
    + escape(", ".join(unavailable)) + '.</p>'
) if unavailable else ""
figure.html.add_child(Element(f'''
<main>
  <header>
    <h1>{escape(TITLE)}</h1>
    <p>How are places to eat, grab a coffee, have a drink and go out distributed across the two areas?</p>
  </header>
  <fieldset id="filters"><legend>Show on both maps</legend>{filters}</fieldset>
  {status}
  <div id="comparison"></div>
  <aside><span class="outline" aria-hidden="true"></span> {WINDOW_METRES / 1000:g} × {WINDOW_METRES / 1000:g} km study windows</aside>
  <footer>{escape(FOOTER)}</footer>
</main>
'''))

# Folium renders each map into a section that has its own place label.
# Both sections have the same width on desktop and stack on narrow screens.
def add_panel(key):
    label = CENTRES[key][0]
    figure.html.add_child(Element(f'''
    <section id="panel-{key}" class="panel" aria-labelledby="label-{key}">
      <h2 id="label-{key}">{escape(label)}</h2>
      <span class="north" aria-label="North">▲<br>N</span>
    </section>'''))
    maps[key].add_to(figure)

add_panel("cabo")
add_panel("manga")
figure.header.add_child(Element('''
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
* { box-sizing: border-box; }
body { margin: 0; color: #112d45; background: #e8f2f8;
       font-family: system-ui, sans-serif; }
main { max-width: 1536px; margin: 0 auto; padding: 24px clamp(14px, 3vw, 44px) 14px; }
header { border-top: 4px solid #173c52; padding-top: 16px; text-align: center; }
h1 { font-family: Georgia, serif; font-size: clamp(1.65rem, 3.7vw, 3.25rem);
     line-height: 1.08; margin: 0; letter-spacing: -.035em; }
header p { margin: 8px 0 14px; font-size: clamp(.9rem, 1.5vw, 1.08rem); }
fieldset { display: flex; justify-content: center; gap: 4px 16px; flex-wrap: wrap; width: fit-content; max-width: 100%;
           min-width: 0; margin: 0 auto 12px;
           padding: 4px 10px 6px; border: 1px solid #92aebb; border-radius: 6px; }
legend { margin-inline: auto; padding: 0 6px; font-size: .85rem; font-weight: 700; }
label { display: inline-flex; align-items: center; gap: 7px; min-height: 40px;
        padding: 0 6px; cursor: pointer; font-weight: 600; font-size: .92rem; }
input { width: 17px; height: 17px; accent-color: #173c52; margin: 0; }
.swatch { width: 12px; height: 12px; border-radius: 100%; background: var(--marker); }
#comparison { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr));
  gap: 10px; height: clamp(395px, 66vh, 770px); }
.panel { position: relative; min-width: 0; min-height: 0; isolation: isolate;
         border: 1px solid #92aebb; border-radius: 8px; overflow: hidden; }
.panel .folium-map { background: #f1eee5; }
.panel .leaflet-control-attribution, .panel .leaflet-control-scale { font-size: 11px; }
.panel h2, .north { position: absolute; z-index: 700; margin: 0; pointer-events: none;
                    background: #f7fbfd; border-radius: 4px; }
.panel h2 { top: 10px; left: 50%; transform: translateX(-50%); white-space: nowrap; padding: 7px 10px; font-size: .92rem; }
.north { right: 10px; top: 10px; padding: 4px 8px; text-align: center;
         font-size: .72rem; font-weight: 800; line-height: 1; }
aside, footer { font-size: .78rem; margin-top: 10px; text-align: center; }
.outline { display: inline-block; width: 16px; height: 10px; border: 2px dashed #426279; }
footer { border-top: 1px solid #92aebb; padding-top: 10px; }
.notice { font-size: .9rem; font-weight: 700; }
@media (max-width: 700px) {
  main { padding-top: 14px; }
  #filters { position: sticky; top: 0; z-index: 800; background: #e8f2f8; }
  #comparison { grid-template-columns: minmax(0, 1fr); height: auto; }
  .panel { height: clamp(320px, 55svh, 490px); }
}
</style>
'''))

# Folium assigns identifiers at render time. Refer to its real variables
# directly so the filter state and resizing always act on the visible maps.
map_entries = ", ".join(f'{key}: {panel.get_name()}' for key, panel in maps.items())
layer_entries = ", ".join(
    f'{category}: [' + ", ".join(groups[key][category].get_name() for key in maps) + "]"
    for category in categories
)
figure.script.add_child(Element(f'''
document.addEventListener("DOMContentLoaded", () => {{
const maps = {{{map_entries}}};
const narrowStartZoom = {MOBILE_ZOOM};
const sharedLayers = {{{layer_entries}}};
'''))
figure.script.add_child(Element(r'''
const comparison = document.querySelector('#comparison');
for (const [key, map] of Object.entries(maps)) {
    const panel = document.querySelector('#panel-' + key);
    panel.appendChild(map.getContainer());
    comparison.appendChild(panel);
    // Fit more of each study window at startup on narrow screens.
    // Later resizes preserve the visitor's own zoom and centre.
    if (matchMedia('(max-width: 700px)').matches) {
        map.invalidateSize({pan: true, animate: false});
        map.setZoom(narrowStartZoom, {animate: false});
    }
    L.control.zoom({position: 'bottomright'}).addTo(map);
    L.control.scale({imperial: false}).addTo(map);
    map.attributionControl.addAttribution(
        '<a href="https://openfreemap.org/">OpenFreeMap</a> · ' +
        '© <a href="https://openmaptiles.org/">OpenMapTiles</a> · ' +
        '© <a href="https://www.openstreetmap.org/copyright">OpenStreetMap contributors</a>'
    );
    // Wait for the resized container and let Leaflet offset the map pane.
    // pan:true keeps the geographic centre fixed while the pixel width changes.
    const observer = new ResizeObserver(() => {
        map.invalidateSize({pan: true, animate: false, debounceMoveend: true});
    });
    observer.observe(panel);
}

document.querySelectorAll('#filters input').forEach(input => {
    const apply = () => {
        Object.values(maps).forEach((map, index) => {
            const group = sharedLayers[input.value][index];
            if (input.checked) map.addLayer(group);
            else map.removeLayer(group);
        });
    };
    input.addEventListener('change', apply);
    apply();
});

});
'''))

In [7]:
# Export the full comparison as a static GitHub Pages entry point.
# Folium embeds the venue data, styles and interactions in this single file.
# The export also embeds the local basemap and the browser libraries.
destination = Path("index.html")
figure.save(destination)
print(f"Saved {destination}")
display(Markdown("[Open the interactive comparison](index.html)"))

Saved index.html


[Open the interactive comparison](index.html)

## Sources and limitations

- Café snapshots were extracted with OSMnx from [OpenStreetMap API map extracts](https://wiki.openstreetmap.org/wiki/API_v0.6#Retrieving_map_data_by_bounding_box:_GET_/api/0.6/map), then restricted to the same study windows. Each snapshot records its source.
- Venue geometry and names © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright), ODbL. The local basemap uses OpenStreetMap data from [OpenFreeMap](https://openfreemap.org/) in the [OpenMapTiles](https://openmaptiles.org/) schema. The background covers the study areas and their immediate surroundings.
- The POI workflow is inspired by Milan Janosov’s [Interactive POI Visualization in Python with OpenStreetMap Data](https://www.youtube.com/watch?v=SyOF6TWFnvA).
- [Cala del Pino and Plaza Bohemia](https://www.lamangaconsorcio.es/la-manga-cala-del-pino.pdf) help locate the southern La Manga study window.

Each window covers the same total area, including water. The comparison shows the distribution of mapped restaurants, cafés, bars, pubs and nightclubs within these windows. OSM coverage and tagging can vary, and venue records may be out of date. The count table reports the snapshot used for the map. The four filters remain independent. The Nightclubs layer is empty in any study window without a venue tagged as a nightclub.